In [ ]:
import pymc as pm
import pandas as pd
import numpy as np
import arviz as az

df = pd.read_csv('full_data.csv', keep_default_na=False, na_values=[''])
df

## Adding data

In [ ]:
stateAbbri = {"Alabama": "AL",
             "Alaska": "AK",
             "Arizona": "AZ",
             "Arkansas": "AR",
             "California": "CA",
             "Colorado": "CO",
             "Connecticut": "CT",
             "Delaware": "DE",
             "Florida": "FL",
             "Georgia": "GA",
             "Hawaii": "HI",
             "Idaho": "ID",
             "Illinois": "IL",
             "Indiana": "IN",
             "Iowa": "IA",
             "Kansas": "KS",
             "Kentucky": "KY",
             "Louisiana": "LA",
             "Maine": "ME",
             "Maryland": "MD",
             "Massachusetts": "MA",
             "Michigan": "MI",
             "Minnesota": "MN",
             "Mississippi": "MS",
             "Missouri": "MO",
             "Montana": "MT",
             "Nebraska": "NE",
             "Nevada": "NV",
             "New Hampshire": "NH",
             "New Jersey": "NJ",
             "New Mexico": "NM",
             "New York": "NY",
             "North Carolina": "NC",
             "North Dakota": "ND",
             "Ohio": "OH",
             "Oklahoma": "OK",
             "Oregon": "OR",
             "Pennsylvania": "PA",
             "Rhode Island": "RI",
             "South Carolina": "SC",
             "South Dakota": "SD",
             "Tennessee": "TN",
             "Texas": "TX",
             "Utah": "UT",
             "Vermont": "VT",
             "Virginia": "VA",
             "Washington": "WA",
             "West Virginia": "WV",
             "Wisconsin": "WI",
             "Wyoming": "WY",   
             "District of Columbia": "DC",
             "American Samoa": "AS",
             "Guam": "GU",
             "Northern Mariana Islands": "MP",
             "Puerto Rico": "PR",
             "U.S. Virgin Islands": "VI",
             "Federated States of Micronesia": "FM",
             "Marshall Islands": "MH",
             "Palau": "PW",
             "Armed Forces Americas": "AA",
             "Armed Forces Europe": "AE",
             "Armed Forces Pacific": "AP",
             "Alberta": "AB",
             "British Columbia": "BC"}
    
data_2023 = pd.read_csv('total_2023.csv')
data_2022 = pd.read_csv('total_2022.csv')
data_2021 = pd.read_csv('total_2021.csv')
data_2020 = pd.read_csv('total_2020.csv')

data_2023.drop(columns=['unitid', 'HD2023.Bureau of Economic Analysis (BEA) regions'], inplace=True)

states_to_remove = ['Armed Forces Americas', 'Armed Forces Europe', 'Armed Forces Pacific',
                    'American Samoa', 'Guam', 'Northern Marianas', 'Puerto Rico',
                    'Federated States of Micronesia', 'Palau', 'Virgin Islands',
                    'Marshall Islands']
data_2023.rename(columns={
    'HD2023.State abbreviation': 'state',
    'institution name': 'name'
}, inplace=True)
data_2023 = data_2023[~data_2023['state'].isin(states_to_remove)]
data_2023['stateAbbr'] = data_2023['state'].map(stateAbbri)

data_2022.rename(columns={
    'HD2022.State abbreviation': 'state',
    'institution name': 'name'
}, inplace=True)
data_2022 = data_2022[~data_2022['state'].isin(states_to_remove)]
data_2022['stateAbbr'] = data_2022['state'].map(stateAbbri)

data_2021.rename(columns={
    'HD2021.State abbreviation': 'state',
    'institution name': 'name'
}, inplace=True)
data_2021 = data_2021[~data_2021['state'].isin(states_to_remove)]
data_2021['stateAbbr'] = data_2021['state'].map(stateAbbri)

data_2020.rename(columns={
    'HD2020.State abbreviation': 'state',
    'institution name': 'name'
}, inplace=True)
data_2020 = data_2020[~data_2020['state'].isin(states_to_remove)]
data_2020['stateAbbr'] = data_2020['state'].map(stateAbbri)

stateRegions = {
    "Alabama": "South",
    "Alaska": "West",
    "Arizona": "West",
    "Arkansas": "South",
    "California": "West",
    "Colorado": "West",
    "Connecticut": "Northeast",
    "Delaware": "South",
    "Florida": "South",
    "Georgia": "South",
    "Hawaii": "West",
    "Idaho": "West",
    "Illinois": "Midwest",
    "Indiana": "Midwest",
    "Iowa": "Midwest",
    "Kansas": "Midwest",
    "Kentucky": "South",
    "Louisiana": "South",
    "Maine": "Northeast",
    "Maryland": "South",
    "Massachusetts": "Northeast",
    "Michigan": "Midwest",
    "Minnesota": "Midwest",
    "Mississippi": "South",
    "Missouri": "Midwest",
    "Montana": "West",
    "Nebraska": "Midwest",
    "Nevada": "West",
    "New Hampshire": "Northeast",
    "New Jersey": "Northeast",
    "New Mexico": "West",
    "New York": "Northeast",
    "North Carolina": "South",
    "North Dakota": "Midwest",
    "Ohio": "Midwest",
    "Oklahoma": "South",
    "Oregon": "West",
    "Pennsylvania": "Northeast",
    "Rhode Island": "Northeast",
    "South Carolina": "South",
    "South Dakota": "Midwest",
    "Tennessee": "South",
    "Texas": "South",
    "Utah": "West",
    "Vermont": "Northeast",
    "Virginia": "South",
    "Washington": "West",
    "West Virginia": "South",
    "Wisconsin": "Midwest",
    "Wyoming": "West",
    "District of Columbia": "South",
    "American Samoa": "West",
    "Guam": "West",
    "Northern Mariana Islands": "West",
    "Puerto Rico": "South"
}

for i in [data_2023, data_2022, data_2021, data_2020]:
    i['region'] = i['state'].map(stateRegions)

temp_state = pd.DataFrame()
temp_region = pd.DataFrame()

for year, data in zip([2020, 2021, 2022, 2023], [data_2020, data_2021, data_2022, data_2023]):
    temp_state[f'totalByState_{year}'] = data.groupby('state')['name'].count()
    temp_region[f'totalByRegion_{year}'] = data.groupby('region')['name'].count()

for year in [2024, 2025]:
    temp_state[f'totalByState_{year}'] = temp_state[f'totalByState_{year-1}']
    temp_region[f'totalByRegion_{year}'] = temp_region[f'totalByRegion_{year-1}']

temp_region

In [ ]:
temp_state

In [ ]:
closed_colleges = df.copy()

for year in [2020, 2021, 2022, 2023, 2024, 2025]:
    temp_state[f'closedByState_{year}'] = closed_colleges[closed_colleges['yearClosed'] == year].groupby('state')['name'].count()
    temp_region[f'closedByRegion_{year}'] = closed_colleges[closed_colleges['yearClosed'] == year].groupby('region')['name'].count()

temp_state.fillna(0, inplace=True)
temp_region.fillna(0, inplace=True)

temp_region = temp_region.astype(int)
temp_state = temp_state.astype(int)

temp_region

In [ ]:
for year in [2020, 2021, 2022, 2023, 2024, 2025]:
    df = df.merge(temp_state[[f'totalByState_{year}', f'closedByState_{year}']], on='state', how='left')
    df = df.merge(temp_region[[f'totalByRegion_{year}', f'closedByRegion_{year}']], on='region', how='left')

df.to_csv('full_data_bhm1.csv', index=False)
df

## BHM 1

In [ ]:
df = pd.read_csv('full_data_bhm1.csv', keep_default_na=False, na_values=[''])

years = [i for i in range(2020, 2026)]
states = df['state'].unique()
regions = df['region'].unique()

print(f"Years: {years}\nStates: {states}\nRegions: {regions}")
df

In [ ]:
state_data = []
region_data = []
for year in years:
    for state in states:
        state_df = df[df['state'] == state]
        total = state_df[f'totalByState_{year}'].sum()
        closed = state_df[f'closedByState_{year}'].sum()
        if total > 0:
            state_data.append({'state': state, 'year': year, 'total': total, 'closed': closed})
    for region in regions:
        region_df = df[df['region'] == region]
        total = region_df[f'totalByRegion_{year}'].sum()
        closed = region_df[f'closedByRegion_{year}'].sum()
        if total > 0:
            region_data.append({'region': region, 'year': year, 'total': total, 'closed': closed})

state_df = pd.DataFrame(state_data)
region_df = pd.DataFrame(region_data)

state_df

## methodology

- First we define our model
- Then we specify our global mean hyperprior
- We define the hyperprior for the variability between regions
- We allow each region to have its own effect, by drawing from a normal distribution centered on the global mean
- Then we map regions to indices for indexing effects
- We define the mapping from states to regions and index the region effects based on each state's region


## variables

- (mu_global) hyperprior for global mean -> a weak prior centered at 0 with a wide var
- (sigma_region) hyperprior for the variability between regions -> this controls how much regions differ from the global mean
- (region_effect) prior for region-specific effects -> allowing each region to deviate from the global mean
- (region_idx_map) mapping from region names to indices for indexing region effects
- (state_to_region) mapping from state to its region
- (state_region_idx) index for region effects based on each state's region
- (sigma_state) hyperprior for the variability between states within regions
- (state_effect) prior for state-specific effects, informed by the region's effect -> each state's effect is drawn from a normal distribution centered on its region's effect
- (logit_p) global mean and state effects to get the logit of the closure probability
- (p) logit to probability using the sigmoid function
- (y = likelihood) model the number of closures as a binomial distribution -> 'y' is the observed number of closures, 'n' is the total number of institutions, 'p' is the probability
- (trace) MCMC sampling

In [ ]:
with pm.Model() as model:
    mu_global = pm.Normal('mu_global', mu=0, sigma=10)
    sigma_region = pm.HalfNormal('sigma_region', sigma=5)
    region_effects = pm.Normal('region_effects', mu=0, sigma=sigma_region, shape=len(regions))
    
    region_idx_map = {region: i for i, region in enumerate(regions)}

    state_to_region = dict(zip(df['state'], df['region']))
    state_region_idx = [region_idx_map[state_to_region[state_df.loc[i, 'state']]] for i in range(len(state_df))]
    
    sigma_state = pm.HalfNormal('sigma_state', sigma=5)
    state_effects = pm.Normal('state_effects', mu=region_effects[state_region_idx], sigma=sigma_state, shape=len(state_df))
    
    logit_p = mu_global + state_effects
    p = pm.Deterministic('p', pm.math.sigmoid(logit_p))
    y = pm.Binomial('y', n=state_df['total'], p=p, observed=state_df['closed'])
    
    trace = pm.sample(return_inferencedata=True, random_seed=42)

az.to_netcdf(trace, 'bhm/trace_bhm_1.nc')
az.summary(trace)

In [ ]:
with pm.Model() as model:
    mu_global = pm.Normal('mu_global', mu=0, sigma=10)
    sigma_region = pm.HalfNormal('sigma_region', sigma=5)
    region_effects = pm.Normal('region_effects', mu=0, sigma=sigma_region, shape=len(regions))
    
    region_idx_map = {region: i for i, region in enumerate(regions)}

    state_to_region = dict(zip(df['state'], df['region']))
    state_region_idx = [region_idx_map[state_to_region[state_df.loc[i, 'state']]] for i in range(len(state_df))]
    
    sigma_state = pm.HalfNormal('sigma_state', sigma=5)
    state_effects = pm.Normal('state_effects', mu=region_effects[state_region_idx], sigma=sigma_state, shape=len(state_df))
    
    logit_p = mu_global + state_effects
    p = pm.Deterministic('p', pm.math.sigmoid(logit_p))
    y = pm.Binomial('y', n=state_df['total'], p=p, observed=state_df['closed'])

    trace = pm.sample(4000, tune=2000, target_accept=0.9, return_inferencedata=True, random_seed=42)

az.to_netcdf(trace, 'bhm/trace_bhm_2.nc')
az.summary(trace)